# Trigger Azure Machine Learning jobs with GitHub Actions

## Introduction

Imagine you're a machine learning engineer, working together with a data science team on a diabetes classification model. The workflow created by the data science team preprocesses data and trains the model. You want to automatically execute the workflow. By doing so, you'll enable automated training (and retraining) of the classification model in different environments, driven by different events.

Automation is an important part in machine learning operations (MLOps). Similar to DevOps, MLOps allows for rapid development and delivery of machine learning artifacts to consumers of those artifacts. An effective MLOps strategy allows for the creation of automated workflows to train, test, and deploy machine learning models while also ensuring model quality is maintained.

Using GitHub Actions, you'll automatically execute an Azure Machine Learning job to train a model. To execute your Azure Machine Learning jobs with GitHub Actions, you'll save your Azure credentials as a secret in GitHub. You'll then define the GitHub Action using YAML.

### Learning objectives

In this module, you'll learn how to:
- Create and assign a service principal the permissions needed to run an Azure Machine Learning job.
- Store Azure credentials securely using secrets in GitHub.
- Create a GitHub Action using YAML that uses the stored Azure credentials to run an Azure Machine Learning job.

## Understand the business problem

You work at Proseware, a young start-up, aiming to improve health care. Together with the data science team, you've recently finished work on operationalizing a diabetes classification model. In other words, you've converted notebooks to scripts that you can execute as an Azure Machine Learning job.

During a presentation of the end-to-end solution to the business and technical stakeholders at Proseware, several questions came up around how to scale the use of this model both from a model creation standpoint and from a consumption standpoint.

In health care, many models use medical data of patients to predict diseases. From previous projects, we've learned that these models are often highly dependent on the geographical location of the population the model is trained on. To make this model scalable, we need to ensure that different versions of the model can automatically be trained based on different data segments.

In the meeting, the business and technical stakeholders have decided to implement a machine learning operations (MLOps) strategy to allow for the rapid creation, update, and deployment of models such as the classification model the data science team has developed for the practitioner web app.

As Proseware uses GitHub to version control its code, the decision was made to use GitHub Actions as the automation component of the MLOps strategy.

The first step in implementing the automation process is to develop a GitHub Action to train the diabetes classification model using Azure Machine Learning jobs.

To create the GitHub Action to trigger model training using Azure Machine Learning compute, you’ll want to:
- Create a service principal using the Azure CLI.
- Store the credentials of the service principal as a secret in GitHub.
- Create a GitHub Action to train the model using Azure Machine Learning compute.

## Explore the solution architecture

It’s important to understand the overall picture before moving ahead with the implementation to ensure all the requirements are met. We also want to ensure the approach is easily adaptable in the future. The focus of this exercise is to start to use GitHub Actions as the orchestration and automation tool for the machine learning operations (MLOps) strategy defined in the solution architecture.

![Explore the solution architecture](../../images/operationalize-ml-models-mlops/explore-the-solution-architecture.png)

<div class="alert alert-info">
<b>ℹ️Note:</b> 

The diagram is a simplified representation of a MLOps architecture. Explore the various use cases in the [MLOps (v2) solution accelerator](https://github.com/Azure/mlops-v2), for more detailed architecture.
</div>

The architecture includes:
1. Setup: Create all necessary Azure resources for the solution.
2. Model development (inner loop): Explore and process the data to train and evaluate the model.
3. Continuous integration: Package and register the model.
4. Model deployment (outer loop): Deploy the model.
5. Continuous deployment: Test the model and promote to production environment.
6. Monitoring: Monitor model and endpoint performance.

Specifically, we’re going to be automating the training portion of the model development, or inner loop, which will ultimately allow us to quickly train and register multiple models for deployment to staging and production environments.

The Azure Machine Learning workspace, Azure Machine Learning compute, and GitHub repository have all been created for you by the infrastructure team.

In addition, the code to train the classification model is production-ready and the data needed to train the model is available in an Azure Blob Storage connected to the Azure Machine Learning workspace.

Your implementation will enable the move from inner to outer loop to be an automated process that happens whenever a data scientist pushes new model code to the GitHub repository, enabling the continuous delivery of machine learning models to downstream consumers of the model, like the web application that will use the diabetes classification model.

## Use GitHub Actions for model training

GitHub Actions is a platform that allows you to automate tasks triggered by events that occur within a GitHub repository. A GitHub Actions workflow consist of jobs. A job groups a set of steps that you can define. One of these steps can use the CLI (v2) to run an Azure Machine Learning job to train a model.

To automate model training with GitHub Actions, you'll need to:
- Create a service principal using the Azure CLI.
- Store the Azure credentials in a GitHub secret.
- Define a GitHub Action in YAML.

### Create a service principal

When you use GitHub Actions to automate Azure Machine Learning jobs, you need to use a service principal to authenticate GitHub to manage the Azure Machine Learning workspace. For example, to train a model using Azure Machine Learning compute, you or any tool that you use, needs to be authorized to use that compute.


<div class="alert alert-info">
<b>💡Tip:</b> 

[Use GitHub Actions to connect to Azure](https://learn.microsoft.com/en-us/azure/developer/github/connect-from-azure).
</div>


### Store the Azure credentials

The Azure credentials you need to authenticate should not be stored in your code or plain text and should instead be stored in a GitHub secret.

To add a secret to your GitHub repository:
1. Navigate to the Settings tab.

![Store the azure credentials 01](../../images/operationalize-ml-models-mlops/store-the-azure-credentials-01.png)


2. In the Settings tab, under Security, expand the Secrets option and select Actions.

![Store the azure credentials 02](../../images/operationalize-ml-models-mlops/store-the-azure-credentials-02.png)

3. Enter your Azure credentials as a secret and name the secret `AZURE_CREDENTIALS`.
4. To use a secret containing Azure credentials in a GitHub Action, refer to the secret in the YAML file.

```yaml
on: [push]

name: Azure Login Sample

jobs:
  build-and-deploy:
    runs-on: ubuntu-latest
    steps:
      - name: Log in with Azure
        uses: azure/login@v1
        with:
          creds: '${{secrets.AZURE_CREDENTIALS}}'
```

### Define the GitHub Action

To define a workflow, you'll need to create a YAML file. You can trigger the workflow to train a model manually or with a push event. Manually triggering the workflow is ideal for testing, while automating it with an event is better for automation.

To configure a GitHub Actions workflow so that you can trigger it manually, use `on: workflow_dispatch`. To trigger a workflow with a push event, use `on: [push]`.

Once the GitHub Actions workflow is triggered, you can add various steps to a job. For example, you can use a step to run an Azure Machine Learning job:

```yaml
name: Manually trigger an Azure Machine Learning job

on:
  workflow_dispatch:

jobs:
  train-model:
    runs-on: ubuntu-latest
    steps:
    - name: Trigger Azure Machine Learning job
      run: |
        az ml job create --file src/job.yml
```

<div class="alert alert-info">
<b>💡Tip:</b> 

More about [GitHub Actions, including core concepts and essential terminology](https://docs.github.com/actions/learn-github-actions/understanding-github-actions).
</div>



## Summary

In this module, you've learned how to:
- Create and assign a service principal the permissions needed to run an Azure Machine Learning job.
- Store Azure credentials securely using secrets in GitHub Secrets.
- Create a GitHub Action using YAML that uses the stored Azure credentials to run an Azure Machine Learning job.